In [0]:
import os
os.environ['https_proxy'] = ''
import pyodbc

class SqlDbUtility:

  def __init__(self, driverString, serverName, port, databaseName, username, password, authDetails={'authentication_type':'SQLAuthentication'}):
    try:
      self.moduleName = "SQL_DB_UTILITY_MODULE"
      self.authDetails = authDetails
      self.driverString = driverString
      self.serverName = serverName
      self.port = port
      self.databaseName = databaseName
      self.id = username
      self.secret = password
      self.jdbcUrl = f"jdbc:sqlserver://{self.serverName}:{self.port};database={self.databaseName};encrypt=true;trustServerCertificate=true"

      if authDetails['authentication_type'] == 'SQLAuthentication':
        self.autenticationType = 'SQLAuthentication'
        self.connectionObject = pyodbc.connect(
          f'DRIVER={{{self.driverString}}};SERVER={self.serverName},{self.port};DATABASE={self.databaseName};UID={self.id};PWD={self.secret};Encrypt=yes;TrustServerCertificate=yes'
        )

      elif authDetails['authentication_type'] == 'ActiveDirectoryServicePrincipal':
        self.autenticationType = 'ActiveDirectoryServicePrincipal'
        self.connectionObject = pyodbc.connect(
          f'DRIVER={{{self.driverString}}};SERVER={self.serverName},{self.port};DATABASE={self.databaseName};Authentication={self.autenticationType};UID={self.id};PWD={self.secret};Encrypt=yes;TrustServerCertificate=yes'
        )

      else:
        raise Exception(f"Invalid Parameters for SQL DB Connection!!")
    except Exception as e:
      print(f"Exception while creating connection to SQL DB: {str(e)}")
      raise

  def executeDmlQuery(self, query):
    try:
      result = None
      print(f"{self.moduleName} | -------------------Start Of DML Query Execution -----------------------")
      print(f"{self.moduleName} | Query To Be Executed --> {query}")

      self.connectionObject.autocommit = True
      cursor = self.connectionObject.cursor()

      for i in range(0, 2):
        try:
          self.connectionObject.autocommit = True
          cursor = self.connectionObject.cursor()
          result = cursor.execute(query)
          break
        except pyodbc.Error as pe:
          if pe.args[0] == "08S01":  # Communication error.
            print("RETRYING THE CONNECTION")
            try:
              self.connectionObject.close()
            except:
              print(f"EXCEPTION WHILE CLOSING CONNECTION | PROCEEDING TO OPEN NEW CONNECTION | ERROR : {pe}")

            if self.autenticationType == 'SQLAuthentication':
              self.connectionObject = pyodbc.connect(
                f'DRIVER={{{self.driverString}}};SERVER={self.serverName},{self.port};DATABASE={self.databaseName};UID={self.id};PWD={self.secret};Encrypt=yes;TrustServerCertificate=yes'
              )
            elif self.autenticationType == 'ActiveDirectoryServicePrincipal':
              self.connectionObject = pyodbc.connect(
                f'DRIVER={{{self.driverString}}};SERVER={self.serverName},{self.port};DATABASE={self.databaseName};Authentication={self.autenticationType};UID={self.id};PWD={self.secret};Encrypt=yes;TrustServerCertificate=yes'
              )
            else:
              print("INVALID AUTHENTICATION")
          else:
            print(pe)
            raise Exception(pe)
        except Exception:
          raise
      print(f"{self.moduleName} | Query Executed Successfully ")
      print(f"{self.moduleName} | -------------------End Of DML Query Execution -----------------------")
      return result
    except Exception:
      print(f"{self.moduleName} | Exception while executing the query ")
      raise

  def executeSelectQuery(self, query):
    try:
      if self.autenticationType == 'SQLAuthentication':
        resultDf = spark.read \
          .format("jdbc") \
          .option("url", self.jdbcUrl) \
          .option("query", query) \
          .option("user", self.id) \
          .option("password", self.secret) \
          .load()

      elif self.autenticationType == 'ActiveDirectoryServicePrincipal':
        resultDf = spark.read \
          .format("jdbc") \
          .option("url", self.jdbcUrl) \
          .option("query", query) \
          .option("AADSecurePrincipalId", self.id) \
          .option("AADSecurePrincipalSecret", self.secret) \
          .option("authentication", self.autenticationType) \
          .load()

      else:
        raise Exception(f"Invalid Connection for SQL DB !!")
      return resultDf
    except Exception:
      raise

service_principal_secret = str(dbutils.secrets.get('zxlc0266xliippue2key02', 'idwstageqampimssql'))
# 214', BUILD server
# 213', REPORTS server
abcDbObj = SqlDbUtility('ODBC Driver 17 for SQL Server',
        '100.122.41.213',    
        30000,
        'staging_genius_reports_TDO',  #ODS3',    
        'MS_DB_IDWS_QA_MPI',
        service_principal_secret,
        authDetails={'authentication_type': 'SQLAuthentication'}
    )

In [0]:
abcDbObj.executeDmlQuery("exec sp_Build_CorpClaimAmounts")

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> exec sp_Build_CorpClaimAmounts
('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The EXECUTE permission was denied on the object 'sp_Build_CorpClaimAmounts', database 'Staging_genius_reports_TDO', schema 'dbo'. (229) (SQLExecDirectW)")
SQL_DB_UTILITY_MODULE | Exception while executing the query 


---------------------------------------------------------------------------
ProgrammingError                          Traceback (most recent call last)
File <command-4117330969926748>, line 50, in SqlDbUtility.executeDmlQuery(self, query)
     49 cursor = self.connectionObject.cursor()
---> 50 result = cursor.execute(query)
     51 break

ProgrammingError: ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The EXECUTE permission was denied on the object 'sp_Build_CorpClaimAmounts', database 'Staging_genius_reports_TDO', schema 'dbo'. (229) (SQLExecDirectW)")

During handling of the above exception, another exception occurred:

Exception                                 Traceback (most recent call last)
File <command-8619427615330321>, line 1
----> 1 abcDbObj.executeDmlQuery("exec sp_Build_CorpClaimAmounts")

File <command-4117330969926748>, line 72, in SqlDbUtility.executeDmlQuery(self, query)
     70   else:
     71     print(pe)
---> 72     raise Exception(pe)
  

In [0]:
# abcDbObj.executeDmlQuery("sp_rename 'ODS3.dbo.XrefCorpExchangeRates', 'ODS3.dbo.XrefCorpExchangeRates_20260420'")

In [0]:
username = "MS_DB_IDWS_QA_MPI"
password = dbutils.secrets.get(
    scope="zxlc0266xliippue2key02",
    key="idwstageqampimssql"

In [0]:
abcDbObj.executeDmlQuery("drop table z_addressheader_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_ATFDYE_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_BrokerBalance_CalendarYE_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_BrokerBalance_UWYE_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_claimheader_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_claimsectiontoextlink_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_coveragetoextlink_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_deductionpercentages_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_earnedpremiumuk_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_earnedpremiumusa_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_endorsementlog_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_extdetailversion_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_extensiondetail_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_historicalroe_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaCorpClaimPayments_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaCorpReserveDeltaOutwards_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaCorpReserveDeltas_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaInwardsCoveragePremiums_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaQuoteAudit_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaSubmissionAudit_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaTransClaimInwards_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaTransClaimOutwards_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_InformaticaTransPolicyInwards_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_invoices_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_legalname_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_names_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_nameslinkedtoaddresses_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_nameusagetoextlink_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_participationgroups_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_participationgroupversions_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_participations_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_policysectiondetail_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_programlink_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_sectiondetailtounit_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_t_preint_ceded_cctd_xgr_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_terms_deleted_rows_TDO_rename	")
abcDbObj.executeDmlQuery("drop table z_Tfr_QuoteLink_Inc_TDO_Rename	")
abcDbObj.executeDmlQuery("drop table z_Tfr_QuoteMultiYearLink_Inc_TDO_Rename	")
abcDbObj.executeDmlQuery("drop table z_Tfr_QuoteRollForwardLink_Inc_TDO_Rename	")
abcDbObj.executeDmlQuery("drop table z_Tfr_QuoteToQuoteLink_Inc_TDO_Rename	")
abcDbObj.executeDmlQuery("drop table z_transclaiminwards_deleted_rows_TDO_rename	")


SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table z_addressheader_deleted_rows_TDO_rename	
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table z_ATFDYE_TDO_rename	
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table z_BrokerBalance_CalendarYE_TDO_rename	
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution --------------

In [0]:
abcDbObj.executeDmlQuery("sp_rename 'd_broker_08122025','z_d_broker_08122025'")
abcDbObj.executeDmlQuery("sp_rename 'd_claim_08122025','z_d_claim_08122025'")
abcDbObj.executeDmlQuery("sp_rename 'd_narratives_08122025','z_d_narratives_08122025'")
abcDbObj.executeDmlQuery("sp_rename 'd_policy_08122025','z_d_policy_08122025'")
abcDbObj.executeDmlQuery("sp_rename 'd_PolicySectionDetail_20221212','z_d_PolicySectionDetail_20221212'")
abcDbObj.executeDmlQuery("sp_rename 'd_quote_08122015','z_d_quote_08122015'")
abcDbObj.executeDmlQuery("sp_rename 'KPIRunDates_20210112','z_KPIRunDates_20210112'")
abcDbObj.executeDmlQuery("sp_rename 'KPIRunDates_bkp','z_KPIRunDates_bkp'")

                         

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> sp_rename 'd_broker_08122025','z_d_broker_08122025'
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> sp_rename 'd_claim_08122025','z_d_claim_08122025'
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> sp_rename 'd_narratives_08122025','z_d_narratives_08122025'
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Qu

In [0]:
GEN_Data_Validation_Result_IDW_Build
IncrementalDeletesLog
IncrementalInsertsLog
inctables
InwardRatio
item
itemgroup
JournalHeaderFile
JournalLinkFile
KPI_VALUE_STAGE
LoadControl
MISParameters
month_split
OBJECT_STAGE
staging_corpreservedeltas
t_Genius_Load_Type
t_Genius_Transaction_Sequence_History
t_ingest_distribute_job_configs
t_wrk_namedlink
t_wrkr_cctd_areacodes
t_wrkr_cptd_areacodes
t_wrkr_ctd_areacodes
t_wrkr_cvrg_basis_cvrg_cd
t_wrkr_ld_motor
t_wrkr_pd_prior_polnbr
t_wrkr_plcy_details_areacodes
t_wrkr_ptd_areacodes
t_wrkr_temp
t_wrkr_xll_policy
T_XLGR_Stg_Informatica_Proc
TableCheck
TableCheck_Full
TableCheckHistory
tblCasualtyPolicy
TotalPortfolio
wrkr_epi_initial


In [0]:
# abcDbObj.executeDmlQuery("sp_rename 'addresscodes_deleted_rows_TDO_rename','z_addresscodes_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'addressheader_deleted_rows_TDO_rename','z_addressheader_deleted_rows_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'claimheader_deleted_rows_TDO_rename','z_claimheader_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'claimsectiontoextlink_deleted_rows_TDO_rename','z_claimsectiontoextlink_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'coveragetoextlink_deleted_rows_TDO_rename','z_coveragetoextlink_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'deductionpercentages_deleted_rows_TDO_rename','z_deductionpercentages_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'earnedpremiumuk_deleted_rows_TDO_rename','z_earnedpremiumuk_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'earnedpremiumusa_deleted_rows_TDO_rename','z_earnedpremiumusa_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'endorsementlog_deleted_rows_TDO_rename','z_endorsementlog_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'extdetailversion_deleted_rows_TDO_rename','z_extdetailversion_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'extensiondetail_deleted_rows_TDO_rename','z_extensiondetail_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'historicalroe_deleted_rows_TDO_rename','z_historicalroe_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'invoices_deleted_rows_TDO_rename','z_invoices_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'legalname_deleted_rows_TDO_rename','z_legalname_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'names_deleted_rows_TDO_rename','z_names_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'nameslinkedtoaddresses_deleted_rows_TDO_rename','z_nameslinkedtoaddresses_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'nameusagetoextlink_deleted_rows_TDO_rename','z_nameusagetoextlink_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'participationgroups_deleted_rows_TDO_rename','z_participationgroups_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'participationgroupversions_deleted_rows_TDO_rename','z_participationgroupversions_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'participations_deleted_rows_TDO_rename','z_participations_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'policysectiondetail_deleted_rows_TDO_rename','z_policysectiondetail_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'programlink_deleted_rows_TDO_rename','z_programlink_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'sectiondetailtounit_deleted_rows_TDO_rename','z_sectiondetailtounit_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 't_preint_ceded_cctd_xgr_TDO_rename','z_t_preint_ceded_cctd_xgr_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'terms_deleted_rows_TDO_rename','z_terms_deleted_rows_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'Tfr_QuoteLink_Inc_TDO_Rename','z_Tfr_QuoteLink_Inc_TDO_Rename'")
# abcDbObj.executeDmlQuery("sp_rename 'Tfr_QuoteMultiYearLink_Inc_TDO_Rename','z_Tfr_QuoteMultiYearLink_Inc_TDO_Rename'")
# abcDbObj.executeDmlQuery("sp_rename 'Tfr_QuoteRollForwardLink_Inc_TDO_Rename','z_Tfr_QuoteRollForwardLink_Inc_TDO_Rename'")
# abcDbObj.executeDmlQuery("sp_rename 'Tfr_QuoteToQuoteLink_Inc_TDO_Rename','z_Tfr_QuoteToQuoteLink_Inc_TDO_Rename'")
# abcDbObj.executeDmlQuery("sp_rename 'transclaiminwards_deleted_rows_TDO_rename','z_transclaiminwards_deleted_rows_TDO_rename'")

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> sp_rename 'claimheader_deleted_rows_TDO_rename','z_claimheader_deleted_rows_TDO_rename'
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------


In [0]:
# abcDbObj.executeDmlQuery("sp_rename 'InformaticaCorpClaimPayments','z_InformaticaCorpClaimPayments_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'InformaticaCorpReserveDeltaOutwards_TDO_rename','z_InformaticaCorpReserveDeltaOutwards_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'InformaticaCorpReserveDeltas_TDO_rename','z_InformaticaCorpReserveDeltas_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'InformaticaInwardsCoveragePremiums_TDO_rename','z_InformaticaInwardsCoveragePremiums_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'InformaticaQuoteAudit_TDO_rename','z_InformaticaQuoteAudit_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'InformaticaSubmissionAudit_TDO_rename','z_InformaticaSubmissionAudit_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'InformaticaTransClaimInwards_TDO_rename','z_InformaticaTransClaimInwards_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'InformaticaTransClaimOutwards_TDO_rename','z_InformaticaTransClaimOutwards_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'InformaticaTransPolicyInwards_TDO_rename','z_InformaticaTransPolicyInwards_TDO_rename'")

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> sp_rename 'InformaticaCorpReserveDeltaOutwards_TDO_rename','z_InformaticaCorpReserveDeltaOutwards_TDO_rename'
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> sp_rename 'InformaticaCorpReserveDeltas_TDO_rename','z_InformaticaCorpReserveDeltas_TDO_rename'
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> sp_rename 'InformaticaInwardsCoveragePremiums_TDO_rename','z_Info

In [0]:
abcDbObj.executeDmlQuery("drop table t_dl_insured_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_inwardbrokers_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_legalentities_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_mainline_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_pms_ctl_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_policy_maxexposure_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_ptdn_inwardpolicy_country_region_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_reinsurer_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_rev_cds_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_ri_claim_trans_dtl_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_ri_pams_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_ri_prem_trans_dtl_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_ribroker_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_ricontract_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_rilimits_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_dw_etl_batch_current_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_ext_claim_header_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_ext_claimant_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_ceded_claim_claimant_transaction_detail_control_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_ceded_claim_claimant_transaction_detail_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_ceded_policy_accounting_month_summary_control_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_ceded_policy_accounting_month_summary_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_claim_claimant_transaction_detail_control_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_claim_claimant_transaction_detail_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_claim_payment_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_line_of_business_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_policy_accounting_month_summary_control_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_policy_accounting_month_summary_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_policy_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_policy_transaction_detail_new_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_risk_location_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_int_risk_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_mi_lta_program_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_t_trn_idw_generic_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_ceded_claim_claimant_transaction_detail_control_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_ceded_claim_claimant_transaction_detail_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_claim_claimant_transaction_detail_control_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_claim_transaction_detail_control_new_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_claim_transaction_detail_new_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_idw_claimant_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_idw_missinglimits_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_pms_xlgr_ctl_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_riprop_limits_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_t_int_claim_claimant_transaction_detail_control_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_temp_t_int_claim_claimant_transaction_detail_TDO_rename")
abcDbObj.executeDmlQuery("drop table t_dl_trn_organization_TDO_rename")

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table t_dl_insured_TDO_rename
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table t_dl_inwardbrokers_TDO_rename
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table t_dl_legalentities_TDO_rename
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_

In [0]:
# abcDbObj.executeDmlQuery("sp_rename 'BrokerBalance_CalendarYE','z_BrokerBalance_CalendarYE_TDO_rename'")
# abcDbObj.executeDmlQuery("sp_rename 'BrokerBalance_UWYE','z_BrokerBalance_UWYE_TDO_rename'")
abcDbObj.executeDmlQuery("sp_rename 'ATFDYE_TDO_rename','z_ATFDYE_TDO_rename'")

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> sp_rename 'ATFDYE_TDO_rename','z_ATFDYE_TDO_rename'
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------


In [0]:
abcDbObj.executeDmlQuery("truncate table z_UW_Temp_Combined")
abcDbObj.executeDmlQuery("truncate table z_UW_TEMP_Inwards")

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> truncate table z_UW_Temp_Combined
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> truncate table z_UW_TEMP_Inwards
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------


In [0]:
abcDbObj.executeDmlQuery("drop table z_TransClaimStatsOutwards_dup121223")
abcDbObj.executeDmlQuery("drop table z_TransPolicyInwards_02122026")
abcDbObj.executeDmlQuery("drop table z_TransPolicyInwards_12092023")
abcDbObj.executeDmlQuery("drop table z_TransPolicyOutwards_02122026")
abcDbObj.executeDmlQuery("drop table z_TransPolicyOutwards_12092023")

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table z_TransClaimStatsOutwards_dup121223
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table z_TransPolicyInwards_02122026
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table z_TransPolicyInwards_12092023
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -------------------

In [0]:
abcDbObj.executeDmlQuery("drop table z_inwardscoveragedeductibles_20201014")
abcDbObj.executeDmlQuery("drop table z_Temp_InwardsCoverageDeduct4_RITM1164657_20201013")
# abcDbObj.executeDmlQuery("drop table z_CoverageDeductibles_20201014")

SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table z_inwardscoveragedeductibles_20201014
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | -------------------Start Of DML Query Execution -----------------------
SQL_DB_UTILITY_MODULE | Query To Be Executed --> drop table z_Temp_InwardsCoverageDeduct4_RITM1164657_20201013
SQL_DB_UTILITY_MODULE | Query Executed Successfully 
SQL_DB_UTILITY_MODULE | -------------------End Of DML Query Execution -----------------------


In [0]:
%sql
select * from xliidw_uat_lpl.staging_genius.MIS_PolicyStatusCodes

PKStatusCode,StatusDesc,ConditionNbr,ConditionType,Term,DW_Insert_Dt
1,Current,1720318,VAL,@,2026-02-01T10:03:41.439Z
2,Renewed,1720318,VAL,@,2026-02-01T10:03:41.439Z
3,Cancelled mid term,1720318,VAL,@,2026-02-01T10:03:41.439Z
4,Lapsed,1720318,VAL,@,2026-02-01T10:03:41.439Z
5,Not taken up,1720318,VAL,@,2026-02-01T10:03:41.439Z
6,Taken Up,1720318,VAL,@,2026-02-01T10:03:41.439Z
